In [1]:
import torch
import torch.nn as nn

class MyRMSNorm(nn.Module):
    def __init__(self, normalized_shape, eps=1e-6):
        super().__init__()
        if isinstance(normalized_shape, int):
            normalized_shape = (normalized_shape,)
        self.normalized_shape = normalized_shape
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(normalized_shape))

    def forward(self, x):
        dims = tuple(range(-len(self.normalized_shape), 0))
        ms = (x * x).mean(dim=dims, keepdim = True)
        return self.weight * (x * torch.rsqrt(ms + self.eps))

In [2]:
torch.manual_seed(0)
x = torch.randn(2, 4, 8, dtype=torch.float64)
w = torch.randn(8, dtype=torch.float64)
eps = 1e-6

mine = MyRMSNorm(8, eps=eps)
off  = nn.RMSNorm(8, eps=eps)
with torch.no_grad():
    mine.weight.copy_(w); off.weight.copy_(w)

print(torch.allclose(mine(x), off(x), atol=1e-12))   # True

True


c:\Users\king\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\nn\functional.py:2954: UserWarning: Mismatch dtype between input and weight: input dtype = double, weight dtype = float, Cannot dispatch to fused implementation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\layer_norm.cpp:347.)
  return torch.rms_norm(input, normalized_shape, weight, eps)
